# Projeto MEGA - Análise de Volume e ROI
Este notebook identifica as categorias e filiais com maior volume de registros para focar a análise de séries temporais.

In [ ]:
import pandas as pd
import os
import numpy as np

# Configurações
target_file = 'data/database_final.csv'
df = pd.read_csv(target_file, low_memory=False)

print(f"Tamanho do DataFrame original: {df.shape}")
df.head()

## 1. Identificação da Categoria e Filial por Volume
Para garantir que o modelo de IA tenha dados suficientes, vamos escolher a combinação com o **maior número de registros**.

In [ ]:
# Agrupando por Categoria e Filial e contando registros
analise_volume = df.groupby(['CATEGORIA', 'FILIAL']).agg(
    TOTAL_REGISTROS=('SKU', 'count'),
    FATUR_TOTAL=('FATUR_VENDA', 'sum')
).reset_index()

# Simulação de Custos e ROI (Assumindo Custo = 70% do Faturamento)
analise_volume['LUCRO_ESTIMADO'] = analise_volume['FATUR_TOTAL'] * 0.30

# Encontrando a combinação com maior número de registros
melhor_foco = analise_volume.sort_values(by='TOTAL_REGISTROS', ascending=False).iloc[0]

print(f"Categoria selecionada (Maior Volume): {melhor_foco['CATEGORIA']}")
print(f"Filial selecionada (Maior Volume): {melhor_foco['FILIAL']}")
print(f"Total de registros: {melhor_foco['TOTAL_REGISTROS']}")
print(f"Lucro Estimado para este grupo: R$ {melhor_foco['LUCRO_ESTIMADO']:,.2f}")

## 2. Filtragem do DataFrame
Filtrando o dataset para o grupo com maior densidade de dados.

In [ ]:
cat_alvo = melhor_foco['CATEGORIA']
filial_alvo = melhor_foco['FILIAL']

df_filtrado = df[(df['CATEGORIA'] == cat_alvo) & (df['FILIAL'] == filial_alvo)].copy()

print(f"O DataFrame filtrado possui {len(df_filtrado)} registros para {cat_alvo} na unidade {filial_alvo}.")
df_filtrado.head()

## 3. Preparação para Modelo Preditivo (Fase 3)
Agora vamos transformar os dados transacionais em uma série temporal diária e criar variáveis (features) para o modelo.

In [ ]:
# 1. Conversão de Data e Agregação Diária
df_filtrado['DATA_ATEND'] = pd.to_datetime(df_filtrado['DATA_ATEND'])

# Agrupamos por dia e somamos a quantidade vendida (Demanda)
df_treino = df_filtrado.groupby('DATA_ATEND').agg(
    DEMANDA_DIARIA=('QTD_VENDA', 'sum'),
    FATURAMENTO_DIARIO=('FATUR_VENDA', 'sum')
).reset_index()

print(f"Dataset agregado por dia: {df_treino.shape[0]} dias de histórico.")

### Engenharia de Atributos
Criando colunas temporais para ajudar o modelo a captar sazonalidade.

In [ ]:
# Extração de atributos básicos
df_treino['MES'] = df_treino['DATA_ATEND'].dt.month
df_treino['DIA_SEMANA'] = df_treino['DATA_ATEND'].dt.dayofweek
df_treino['EH_FIM_DE_SEMANA'] = df_treino['DIA_SEMANA'].isin([5, 6]).astype(int)

def get_estacao(data):
    mes = data.month
    if mes in [1, 2, 3]: return 'Verao'
    if mes in [4, 5, 6]: return 'Outono'
    if mes in [7, 8, 9]: return 'Inverno'
    return 'Primavera'

df_treino['ESTACAO'] = df_treino['DATA_ATEND'].apply(get_estacao)

# One-Hot Encoding para a Estação (facilita para modelos lineares/árvores)
df_treino = pd.get_dummies(df_treino, columns=['ESTACAO'], prefix='EST')

print("Engenharia de atributos concluída. Dataset pronto para treino:")
df_treino.head()

## 4. Verificação de Integridade
Garantindo que não perdemos volume de vendas na agregação.

In [ ]:
soma_original = df_filtrado['QTD_VENDA'].sum()
soma_agregada = df_treino['DEMANDA_DIARIA'].sum()

print(f"Soma total original: {soma_original:.2f}")
print(f"Soma total agregada: {soma_agregada:.2f}")
print(f"Diferença: {abs(soma_original - soma_agregada):.4f}")